# KOA Semantic Ground (LLM-only, Prolog output)

Este notebook gera o arquivo `KOA_semantic_ground.pl` **somente via LLM**.

Princípios:
- **Nenhuma heurística/regex de matching** fora do LLM.
- O LLM faz: (1) digest do requirements, (2) digest do contrato, (3) geração do Semantic Ground final.
- Saída final: **Prolog direto** (sem JSON).

Pipeline:
1. **Requirements Digest (LLM)**: extrai pares (req_id, question_id), textos, e hints conceituais.
2. **Contract Digest (LLM)**: sumariza cláusulas e obrigações/direitos/poderes relevantes.
3. **Semantic Ground (LLM)**: gera `supports_req/3`, `question_targets_req/2`, `best_clause_for_question/4`, `evidence/6`, `justification/4`, `question_underpinned_by/2`, `bridge_q_to_ufo/2`, `bridge_c_to_ufo/2`, `satisfied/1`.

> Observação: o notebook pode ser executado **de cima para baixo** sem depender de ordem manual.


In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
# Imports
import os
from pathlib import Path
import re
import time

from google import genai
import datetime


In [3]:
# Recupera chave Gemini com secrets do Colab
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get('GOOGLE_API_KEY')

In [4]:
# Configuração de pastas e arquivos (mantendo a estrutura/pastas do projeto)
directory = '/content/drive/MyDrive/KOA/onboarding/modelos/'
persist_directory = '/content/drive/MyDrive/KOA/onboarding/modelos/'
contract_directory = '/content/drive/MyDrive/KOA/onboarding/contratos_COM_UFO/'

REQUIREMENTS_PROLOG_FILE = Path(directory) / "KOA_semantic_requirements_normalized.pl"
CONTRACT_PROLOG_DIR = Path(contract_directory)
CONTRACT_PROLOG_FILES = sorted(CONTRACT_PROLOG_DIR.glob("KOA_UFO_contrato_*.pl"))

print("REQUIREMENTS_PROLOG_FILE:", REQUIREMENTS_PROLOG_FILE)
print("CONTRACT_PROLOG_DIR:", CONTRACT_PROLOG_DIR)
print("Found contract files:", len(CONTRACT_PROLOG_FILES))

REQUIREMENTS_PROLOG_FILE: /content/drive/MyDrive/KOA/onboarding/modelos/KOA_semantic_requirements_normalized.pl
CONTRACT_PROLOG_DIR: /content/drive/MyDrive/KOA/onboarding/contratos_COM_UFO
Found contract files: 64


In [5]:
# Leitura do arquivo de requisitos (contratos serão processados em loop mais adiante)
req_text = REQUIREMENTS_PROLOG_FILE.read_text(encoding="utf-8", errors="ignore")

print("Requirements chars:", len(req_text))


Requirements chars: 15522


In [6]:
# Função de chamada ao Gemini (texto puro)
def call_gemini_text(model: str, prompt: str, max_retries: int = 3, sleep_s: float = 2.0) -> str:
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            client = genai.Client(api_key=os.environ.get('GEMINI_API_KEY') or os.environ.get('GOOGLE_API_KEY'))
            resp = client.models.generate_content(model=model, contents=prompt)
            text = getattr(resp, "text", None) or (resp.candidates[0].content.parts[0].text if getattr(resp, "candidates", None) else None)
            if not text:
                raise ValueError("Gemini retornou resposta vazia.")
            return text.strip()
        except Exception as e:
            last_err = e
            if attempt < max_retries:
                time.sleep(sleep_s)
            else:
                raise

def extract_prolog_block(text: str) -> str:
    # tenta capturar bloco ```prolog ... ``` ou ``` ... ```
    m = re.search(r"```prolog\s*(.*?)```", text, flags=re.DOTALL|re.IGNORECASE)
    if not m:
        m = re.search(r"```\s*(.*?)```", text, flags=re.DOTALL)
    return (m.group(1).strip() if m else text.strip())

def basic_validate_prolog(prolog_text: str) -> None:
    required_markers = ["bridge_q_to_ufo(", "bridge_c_to_ufo(", "question_underpinned_by("]
    missing = [m for m in required_markers if m not in prolog_text]
    if missing:
        print("[WARN] Output não contém todos os predicados esperados:", missing)

gemini_model = "gemini-2.0-flash"
print("Gemini model:", gemini_model)

# -----------------------------
# Post-processing helpers (LLM outputs)
# -----------------------------
DEFAULT_GEMINI_MODEL = "gemini-2.0-flash"

def strip_prolog_fences(text: str) -> str:
    """Remove markdown code fences (``` or ```prolog) and surrounding whitespace."""
    if not text:
        return text
    # Remove opening/closing fences on their own lines
    text = re.sub(r"^```[a-zA-Z0-9_-]*\s*\n", "", text.strip())
    text = re.sub(r"\n```\s*$", "", text.strip())
    # Also remove stray fences anywhere
    text = text.replace("```prolog", "").replace("```", "")
    return text.strip()

def fix_sig_requires_nesting(text: str) -> str:
    # Fix patterns like sig_requires(sig_requires(...)).
    return re.sub(r"\bsig_requires\s*\(\s*sig_requires\s*\(", "sig_requires(", text)


def normalize_alias_lists(text: str) -> str:
    """Ensure alias predicates use list syntax for alternatives (wrap single atoms/strings into [..])."""
    lines = text.splitlines()
    out = []
    # Predicates whose last argument must be a list
    two_arg = {"kb_topic_alias", "kb_action_alias", "kb_relation_alias", "kb_normative_kind_alias",
               "kb_right_alias", "kb_obligation_alias", "kb_permission_alias", "kb_prohibition_alias", "kb_power_alias"}
    for ln in lines:
        s = ln.strip()
        # kb_*_alias(Canonical, Alts).
        m = re.match(r"^(%\s*)?(kb_[a-z_]+_alias)\((.+)\)\.$", s)
        if m and not m.group(1):  # don't touch comment lines
            pred = m.group(2)
            args = m.group(3)
            # Split args at top-level commas (simple heuristic; safe for typical alias facts)
            parts = [p.strip() for p in args.split(",")]
            if pred in two_arg and len(parts) == 2:
                canon, alts = parts
                if not alts.startswith("["):
                    alts = f"[{alts}]"
                out.append(f"{pred}({canon}, {alts}).")
                continue
            # kb_attribute_alias(KeyTag, Canonical, Alts).
            if pred == "kb_attribute_alias" and len(parts) == 3:
                key, canon, alts = parts
                if not alts.startswith("["):
                    alts = f"[{alts}]"
                out.append(f"{pred}({key}, {canon}, {alts}).")
                continue
        out.append(ln)
    return "\n".join([l.rstrip() for l in out]).strip()

def clean_prolog_output(text: str) -> str:
    """Apply all required cleanups to LLM-produced Prolog."""
    text = strip_prolog_fences(text or "")
    text = fix_sig_requires_nesting(text)
    text = normalize_alias_lists(text)
    return text.strip()


Gemini model: gemini-2.0-flash


## LLM-only Semantic Ground — Contract-Independent

Este notebook gera **um único** arquivo Prolog (`KOA_semantic_ground.pl`) que atua como uma **ponte semântica** entre:

- **Semantic Requirements** (question frames → critérios)
- **Regras/assinaturas de matching** (padrões abstratos para grounding em qualquer `contrato.pl`)

O arquivo gerado **NUNCA** inclui:
- ids de contratos (`contrato_*`)
- ids de cláusulas (`clausula_*`)
- fatos instanciados por contrato (`supports_*`, `unmapped_*`, `confidence(...)`)

Na aplicação de Q/A, o prompt do LLM deve conter:
- `KOA_semantic_ground.pl`
- `contrato_X.pl`

e o grounding é feito **on-the-fly** durante a resposta.


In [7]:
# -----------------------------
# PROMPTS (LLM-only; domain-agnostic; Prolog output; NO JSON)
# -----------------------------

CRITERIA_PROMPT = """
You are a Knowledge Engineer specialized in Neuro-Symbolic (Hybrid AI) systems.

You will receive a *Semantic Requirements* file as Prolog text.

GOAL:
Generate a contract-independent KOA *Criteria + Assessment* model in Prolog (NO semantic matching layer yet).
Your output MUST contain, in this EXACT ORDER:

1) METADATA + THRESHOLDS
   - satisfaction_threshold(0.70). (use the threshold provided in the input; if none, keep 0.70)

2) SEMANTIC REQUIREMENTS (ORIGINAL CONTENT)
   - Copy verbatim (same predicate name, same arguments, same ids, same strings) the facts from the input requirements
     that define question frames, categories, and concepts. Do NOT paraphrase.
   - Include at least these predicates if present in the input:
     question_frame/1, frame_question/2, frame_intent/2,
     category/1, category_label/2, category_type/2, category_level/2,
     concept/1, concept_label/2, concept_definition/2,
     and any additional requirement-structure predicates that appear (keep them verbatim).

3) REQUIREMENTS-DERIVED MODEL (THE "REQUIREMENTS MODEL")
   - Derive and output ONLY the following Prolog facts:
     criterion/1,
     req_criterion/2,
     criterion_target/2,
     question_operationalizes_criterion/2.
   - IMPORTANT: req_criterion/2 MUST link each question_frame id (qf_*) to exactly ONE criterion id.
     That is: req_criterion(QfId, CritId) where QfId is a qf_* present in the copied requirements.
     Every question_frame/1 MUST appear in exactly one req_criterion/2 and exactly one question_operationalizes_criterion/2.
     Do NOT use intents (e.g., seguranca_informacao, lgpd) as the first argument of req_criterion/2.
   - IDs: reuse existing ids when available; otherwise create stable ids crit_001, crit_002, ... without gaps.
   - Do NOT introduce any contract ids or clause ids. This file is global.

4) ASSESSMENT MODEL (THE "EVALUATION MODEL")
   - Infer assessment dimensions and levels from the requirements when present (e.g., criticidade levels).
   - Output:
     assessment_dimension/1,
     dimension_criterion/2,
     dimension_level/2,
     level_description/3,
     (optional if supported) level_concept/3, level_rule/3.

   - LEVEL RULES (MANDATORY WHEN LEVELS EXIST):
     If you output dimension_level/2 for a dimension D, you MUST ALSO output level_rule/3 facts for that same D.
     The rule must be inferred from the level_description/3 text by mapping it to one or more criteria ids (from the model you derived).
     Example of structure (do NOT hardcode domain words; infer them from the requirements):
       level_rule(D, Level, [Crit1,Crit2,...]).
     If the requirements do NOT provide enough information to map a level to criteria, you MUST NOT guess:
       - Prefer omitting that level_rule/3, OR output it as an explicit empty list ONLY IF the requirements truly define a "no-access/no-risk" case.

CONSISTENCY / INTEGRITY RULES (MANDATORY):
- Output MUST be Prolog ONLY. You may use Prolog comment lines starting with '%' to separate the 4 sections.
- Never output JSON, markdown, code fences, or explanations.
- Never output any contract-specific id (no contrato_*, koa_ufo_contrato_*, clausula_*, etc).
- Avoid duplicates: emit each unique fact once.
- Do NOT output ANY semantic matching layer facts here (NO criterion_signature/2, NO sig_requires/2, NO kb_*_alias/*).

Now, read the input requirements and produce the Prolog output following the structure above.
""".strip()


SEMANTIC_RULES_PROMPT = """
You are a Knowledge Engineer specialized in Neuro-Symbolic (Hybrid AI) systems.

INPUT
1) Criteria Model (Prolog facts) produced from the Semantic Requirements
2) The Semantic Requirements (Prolog text)

GOAL
Produce ONLY contract-independent semantic matching rules and signatures that define how each criterion
should be grounded into an arbitrary Prolog contract knowledge base at QA time.

IMPORTANT CONSTRAINT (Fix the 'entity_type' issue)
- DO NOT express signatures in terms of domain entity types like prestador_servico/colaborador/etc.
- DO NOT use requires_entity_type/1.
Instead, express signatures in terms of:
  (a) normative kind (obligation|permission|prohibition|right|power)
  (b) semantic topic/action/relation/attribute tags, plus aliases
  (c) optional KB-structure hints (predicate patterns), WITHOUT naming specific contracts or clauses

OUTPUT FORMAT
- Output MUST be Prolog ONLY.
- No markdown, no commentary, no code fences.
- Only facts/rules ending with '.'.

ALLOWED OUTPUT PREDICATES (ONLY these; use what you need)

% Signatures per criterion
criterion_signature(CritId, SigId).

% Requirements that a KB evidence candidate should satisfy (all contract-independent)
sig_requires(SigId, requires_normative_kind(Kind)).           % Kind in {obligation, permission, prohibition, right, power}
sig_requires(SigId, requires_topic(TopicTag)).
sig_requires(SigId, requires_action(ActionTag)).
sig_requires(SigId, requires_relation(RelTag)).
sig_requires(SigId, requires_attribute(KeyTag, ValueTag)).
sig_requires(SigId, requires_kb_predicate(PredSlashArity)).   % ONLY if explicitly supported by requirements

% Aliases to make grounding robust to lexical variation in the contract KB
kb_topic_alias(Canonical, [Alt1,Alt2,...]).
kb_action_alias(Canonical, [Alt1,Alt2,...]).
kb_relation_alias(Canonical, [Alt1,Alt2,...]).
kb_attribute_alias(KeyTag, Canonical, [Alt1,Alt2,...]).
kb_normative_kind_alias(CanonicalKind, [Alt1,Alt2,...]).      % optional, if KB uses alternative labels

% Helper predicates (recommended)
criterion_requires_normative_kind(Crit, Kind) :-
    criterion_signature(Crit, Sig),
    sig_requires(Sig, requires_normative_kind(Kind)).

criterion_requires_topic(Crit, Topic) :-
    criterion_signature(Crit, Sig),
    sig_requires(Sig, requires_topic(Topic)).

criterion_requires_action(Crit, Action) :-
    criterion_signature(Crit, Sig),
    sig_requires(Sig, requires_action(Action)).

criterion_requires_relation(Crit, Rel) :-
    criterion_signature(Crit, Sig),
    sig_requires(Sig, requires_relation(Rel)).

criterion_requires_attribute(Crit, Key, Val) :-
    criterion_signature(Crit, Sig),
    sig_requires(Sig, requires_attribute(Key, Val)).

ALIAS RULES (MANDATORY)
- All kb_*_alias lists MUST contain DOUBLE-QUOTED STRINGS (not atoms), e.g., ["dados pessoais","informações sigilosas","dados sensíveis"].
- Provide at least 3 textual synonyms/variants per canonical tag whenever possible, extracted/inferred from the requirements text.
- Prefer Portuguese variants; include English variants when clearly supported by the requirements.

SIGNATURE ANCHOR RULE (MANDATORY)
Each criterion_signature MUST include at least ONE of:
- sig_requires(SigId, requires_normative_kind(_)).
- sig_requires(SigId, requires_action(_)).
- sig_requires(SigId, requires_relation(_)).
Do NOT produce signatures that rely only on broad topics (e.g., informacao, servico, acesso) with no anchor,
unless the requirements truly provide nothing else.

CRITICAL RULES
- NEVER mention ContractId, contrato_*, or any specific clause/clausula_* identifier.
- NEVER reference any concrete EntityId from any contract KB.
- Focus on abstract patterns only (signatures + aliases + helper rules).
- Do NOT copy any contract text.
- No markdown, no explanations, only Prolog facts/rules ending with '.'.

Now produce ONLY the allowed Prolog predicates.
""".strip()

In [8]:
# -----------------------------
# 1) CRITERIA MODEL (LLM) from requirements
# -----------------------------
criteria_prompt = CRITERIA_PROMPT + "\n\n" + req_text
criteria_model_prolog = clean_prolog_output(call_gemini_text(model=DEFAULT_GEMINI_MODEL, prompt=criteria_prompt))

print("Criteria model chars:", len(criteria_model_prolog))
print(criteria_model_prolog[:800])


Criteria model chars: 19180
% METADATA + THRESHOLDS
satisfaction_threshold(0.70).

% SEMANTIC REQUIREMENTS (ORIGINAL CONTENT)
concept(c_7fbcda231b).
concept_label(c_7fbcda231b, "risco operacional").
concept_definition(c_7fbcda231b, "possibilidade de perdas decorrentes de falhas, inadequações ou eventos externos que afetem processos, pessoas, sistemas ou controles internos").
evidence(c_7fbcda231b, "risco operacional é um conceito mais amplo que se refere à possibilidade de perdas decorrentes de falhas, inadequações ou eventos externos que afetem processos, pessoas, sistemas ou controles internos", "chunk:0").

concept(c_5926e8e1c1).
concept_label(c_5926e8e1c1, "Risco à Segurança da Informação").
concept_definition(c_5926e8e1c1, "potencial de violação da integridade, confidencialidade, disponibilidade ou autenticidade


In [9]:
# -----------------------------
# 2) SEMANTIC MATCHING RULES (LLM) from criteria model + requirements
# -----------------------------
rules_prompt = (
    SEMANTIC_RULES_PROMPT
    + "\n\n% --- CRITERIA MODEL ---\n"
    + criteria_model_prolog.strip()
    + "\n\n% --- SEMANTIC REQUIREMENTS ---\n"
    + req_text
)

semantic_rules_prolog = clean_prolog_output(call_gemini_text(model=DEFAULT_GEMINI_MODEL, prompt=rules_prompt))


In [10]:
# -----------------------------
# 3) SANITY CHECKS (no inference; just validation of constraints)
# -----------------------------
checks = {
    "has_contrato_id": re.search(r"\bcontrato_\w+", semantic_rules_prolog) is not None,
    "has_clausula_id": re.search(r"\bclausula_\w+", semantic_rules_prolog) is not None,
    "has_supports_facts": re.search(r"\bsupports_\w+\(", semantic_rules_prolog) is not None,
    "has_requires_entity_type": re.search(r"requires_entity_type\(", semantic_rules_prolog) is not None,
}

print("Sanity checks:", checks)

# Hard fail if forbidden content appears in semantic rules
if checks["has_contrato_id"] or checks["has_clausula_id"] or checks["has_supports_facts"] or checks["has_requires_entity_type"]:
    raise ValueError("Semantic rules contain forbidden contract-dependent content (contrato_*, clausula_*, supports_* or requires_entity_type).")


Sanity checks: {'has_contrato_id': False, 'has_clausula_id': False, 'has_supports_facts': False, 'has_requires_entity_type': False}


In [11]:
# -----------------------------
# 4) WRITE UNIFIED SEMANTIC GROUND (single Prolog file)
# -----------------------------
def extract_threshold(prolog_text: str, default: float = 0.70) -> float:
    m = re.search(r"\bsatisfaction_threshold\s*\(\s*([0-9]*\.?[0-9]+)\s*\)\s*\.", prolog_text or "")
    if not m:
        return default
    try:
        return float(m.group(1))
    except Exception:
        return default

def remove_satisfaction_threshold_facts(prolog_text: str) -> str:
    lines = (prolog_text or "").splitlines()
    kept = [ln for ln in lines if not re.search(r"\bsatisfaction_threshold\s*\(", ln)]
    return "\n".join(kept).strip()

threshold = extract_threshold(criteria_model_prolog, default=0.70)

ts = datetime.datetime.utcnow().replace(microsecond=0).isoformat() + "Z"
header = (
    "% ===== KOA Semantic Ground (LLM-only, unified; contract-independent; no contract text) =====\n"
    f"% generated_at_utc: {ts}\n"
    f"% requirements_file: {REQUIREMENTS_PROLOG_FILE.name}\n"
    f"% contracts_dir: {str(CONTRACT_PROLOG_DIR)}\n"
    f"% schema: satisfaction_threshold/1, criterion/1, req_criterion/2, criterion_target/2, question_operationalizes_criterion/2,\n"
    f"%         assessment_dimension/1, dimension_criterion/2, dimension_level/2, level_description/3,\n"
    f"%         level_rule/3, criterion_signature/2, sig_requires/2,\n"
    f"%         kb_*_alias/*, criterion_requires_*/*\n"
    f"% ===========================================================================\n\n"
)

# Ensure no markdown fences leak into the final file
criteria_clean = clean_prolog_output(criteria_model_prolog)
rules_clean = clean_prolog_output(semantic_rules_prolog)

out_text = header
out_text += f"satisfaction_threshold({threshold:.2f}).\n\n"

out_text += "% ---- CRITERIA MODEL (from requirements) ----\n"
out_text += remove_satisfaction_threshold_facts(criteria_clean) + "\n\n"
out_text += "% ---- SEMANTIC MATCHING RULES (contract-independent) ----\n"
out_text += rules_clean + "\n"

out_path = Path(persist_directory) / "KOA_semantic_ground.pl"
out_path.write_text(out_text, encoding="utf-8")

print("Wrote:", out_path)
print("Total chars:", len(out_text))


/tmp/ipython-input-1689436959.py:20: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.datetime.utcnow().replace(microsecond=0).isoformat() + "Z"


Wrote: /content/drive/MyDrive/KOA/onboarding/modelos/KOA_semantic_ground.pl
Total chars: 24504
